In [1]:
import os
import numpy as np
import trimesh
import tqdm 

os.makedirs('../results', exist_ok=True)

In [2]:
def uniform_sampling_from_mesh(vertices, faces, sample_num):
    # -------- TODO -----------
    # 1. compute area of each triangles
    # 2. compute probability of each triangles from areas
    # 3. sample N faces according to the probability
    # 4. for each face, sample 1 point
    # Note that FOR-LOOP is not allowed!
    # -------- TODO -----------

    # Step 1 & 2: vectorized area and probability
    v0 = vertices[faces[:, 0]]  # (F, 3)
    v1 = vertices[faces[:, 1]]  # (F, 3)
    v2 = vertices[faces[:, 2]]  # (F, 3)
    cross = np.cross(v1 - v0, v2 - v0)          # (F, 3)
    area = 0.5 * np.linalg.norm(cross, axis=1)  # (F,)
    prob = area / area.sum()                      # (F,)

    # Step 3: sample face indices proportional to area
    face_idx = np.random.choice(len(faces), size=sample_num, p=prob)

    # Step 4: uniform point inside each triangle — Osada et al. sqrt trick
    r1 = np.random.uniform(0, 1, sample_num)
    r2 = np.random.uniform(0, 1, sample_num)
    sqrt_r1 = np.sqrt(r1)
    u = 1.0 - sqrt_r1          # (N,)
    v = sqrt_r1 * (1.0 - r2)   # (N,)
    w = sqrt_r1 * r2            # (N,)

    A = vertices[faces[face_idx, 0]]  # (N, 3)
    B = vertices[faces[face_idx, 1]]  # (N, 3)
    C = vertices[faces[face_idx, 2]]  # (N, 3)
    uniform_pc = u[:, None] * A + v[:, None] * B + w[:, None] * C  # (N, 3)

    return area, prob, uniform_pc

In [3]:
def farthest_point_sampling(pc, sample_num):
    # -------- TODO -----------
    # FOR LOOP is allowed here.
    # -------- TODO -----------
    N = pc.shape[0]
    selected = np.zeros(sample_num, dtype=int)
    selected[0] = np.random.randint(N)
    # dist[i] = squared distance from point i to the nearest already-selected point
    dist = np.full(N, np.inf)

    for i in range(1, sample_num):
        last_pt = pc[selected[i - 1]]                      # (3,)
        d = np.sum((pc - last_pt) ** 2, axis=1)            # (N,)
        dist = np.minimum(dist, d)                          # keep closest-so-far
        selected[i] = np.argmax(dist)                       # pick the farthest

    results = pc[selected]
    return results

In [4]:
# task 1: uniform sampling 

obj_path = 'spot.obj'
mesh = trimesh.load(obj_path)
print('faces shape: ', mesh.faces.shape)
sample_num = 512
area, prob, uniform_pc = uniform_sampling_from_mesh(mesh.vertices, mesh.faces, sample_num)

# Visualization. For you to check your code
np.savetxt('uniform_sampling_vis.txt', uniform_pc)

print('area shape: ',area.shape)
print('prob shape: ',prob.shape)
print('pc shape: ',uniform_pc.shape)
# the result should satisfy: 
#       area.shape = (13712, ) 
#       prob.shape = (13712, ) 
#       uniform_pc.shape = (512, 3) 

# For submission
save_dict = {'area': area, 'prob': prob, 'pc': uniform_pc}
np.save('../results/uniform_sampling_results', save_dict)

faces shape:  (13712, 3)
area shape:  (13712,)
prob shape:  (13712,)
pc shape:  (512, 3)


In [5]:
# task 2: FPS

init_sample_num = 2000
final_sample_num = 512
_,_, tmp_pc = uniform_sampling_from_mesh(mesh.vertices, mesh.faces, init_sample_num)
fps_pc = farthest_point_sampling(tmp_pc, final_sample_num)

# Visualization. For you to check your code
np.savetxt('fps_vis.txt', fps_pc)

# For submission
np.save('../results/fps_results', fps_pc)

In [6]:
# task 3: metrics

import sys, io
from earthmover.earthmover import earthmover_distance   # EMD may be very slow (1~2mins)

def chamfer_distance(pc1, pc2):
    """Symmetric Chamfer Distance (mean L2)."""
    diff = pc1[:, None, :] - pc2[None, :, :]        # (N, M, 3)
    dist2 = np.sum(diff ** 2, axis=-1)               # (N, M)
    cd = (np.mean(np.sqrt(np.min(dist2, axis=1))) +
          np.mean(np.sqrt(np.min(dist2, axis=0))))
    return cd

def compute_emd(pc1, pc2):
    """EMD via earthmover LP; suppress per-flow print output."""
    p1 = [tuple(p) for p in pc1]
    p2 = [tuple(p) for p in pc2]
    buf = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buf
    try:
        val = earthmover_distance(p1, p2)
    finally:
        sys.stdout = old_stdout
    return val

# -----------TODO---------------
# compute chamfer distance and EMD for two point clouds sampled by uniform sampling and FPS.
# sample and compute CD and EMD again. repeat for five times.
# save the mean and var.
# -----------TODO---------------

CD_list  = []
EMD_list = []

for trial in range(5):
    print(f"Trial {trial+1}/5 ...")
    _, _, uni_pc  = uniform_sampling_from_mesh(mesh.vertices, mesh.faces, 512)
    _, _, tmp_big = uniform_sampling_from_mesh(mesh.vertices, mesh.faces, 2000)
    fps_pc = farthest_point_sampling(tmp_big, 512)

    cd  = chamfer_distance(uni_pc, fps_pc)
    emd = compute_emd(uni_pc, fps_pc)
    CD_list.append(cd)
    EMD_list.append(emd)
    print(f"  CD={cd:.6f}  EMD={emd:.6f}")

CD_mean  = float(np.mean(CD_list))
CD_var   = float(np.var(CD_list))
EMD_mean = float(np.mean(EMD_list))
EMD_var  = float(np.var(EMD_list))

print(f"\nCD  mean={CD_mean:.6f}  var={CD_var:.2e}")
print(f"EMD mean={EMD_mean:.6f}  var={EMD_var:.2e}")

# For submission
np.save('../results/metrics', {'CD_mean': CD_mean, 'CD_var': CD_var,
                                'EMD_mean': EMD_mean, 'EMD_var': EMD_var})

Trial 1/5 ...
  CD=2.719233  EMD=2.339563
Trial 2/5 ...
  CD=2.658402  EMD=2.170005
Trial 3/5 ...
  CD=2.832595  EMD=2.723345
Trial 4/5 ...
  CD=2.740911  EMD=2.304195
Trial 5/5 ...
  CD=2.740238  EMD=2.481014

CD  mean=2.738276  var=3.13e-03
EMD mean=2.403625  var=3.54e-02
